# 🎙️ TORGO Dysarthric ASR Pipeline — Inference & Evaluation

**Pipeline Architecture:**
```
Audio → ruch9265/distil-whisper-torgo → Pause Formatter → [LLM Gate] → LLaMA-3.3-70B (Groq) → Final Transcript
```

**Evaluation Phases:**
| Phase | What We Measure |
|---|---|
| Phase 1: Baseline Whisper (tiny base) | Raw WER/CER before fine-tuning |
| Phase 2: Fine-tuned Whisper | WER/CER improvement from fine-tuning |
| Phase 3: Fine-tuned + LLM Repair | Final WER/CER/BERTScore/Semantic Sim |

**Dataset:** `abnerh/TORGO-database` — 16,552 samples, 16kHz English dysarthric speech  
**GPU:** Kaggle T4 (15GB VRAM)  
**Runtime estimate:** ~2.5h full dataset, ~8min subset (200 samples)

---
> ⚠️ **Kaggle Secrets needed:** `HF_TOKEN`, `GROQ_API_KEY`

## 📦 Cell 1 — Install Dependencies

In [1]:
# ─────────────────────────────────────────────────────────────────
# Install all required packages
# Using --quiet to reduce output noise on Kaggle
# ─────────────────────────────────────────────────────────────────
import subprocess, sys

packages = [
    # Core ML
    "transformers>=4.40.0",
    "datasets>=2.18.0",
    "accelerate>=0.28.0",
    "torch>=2.2.0",           # Already on Kaggle T4 image, but pin for safety
    "torchaudio",
    # Evaluation metrics
    "jiwer",                  # WER / CER / MER computation
    "bert-score",             # BERTScore (semantic F1)
    "sentence-transformers",  # Semantic similarity (cosine)
    "evaluate",               # HF evaluate for cleaner metric API
    # LLM / API
    "groq",                   # Groq Python SDK (llama-3.3-70b)
    "langchain-core",
    "langchain-groq",
    # Utilities
    "huggingface_hub",
    "soundfile",
    "librosa",                # Audio preprocessing / resampling
    "pandas",
    "tqdm",
    "psutil",                 # Memory monitoring
]

for pkg in packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        check=False  # Don't crash on minor version conflicts
    )

print("✅ All packages installed.")

✅ All packages installed.


## ⚙️ Cell 2 — Imports & Environment Setup

In [2]:
# ─────────────────────────────────────────────────────────────────
# Core imports. Order matters: env vars BEFORE huggingface imports.
# ─────────────────────────────────────────────────────────────────
import os, gc, re, json, time, warnings, logging
from pathlib import Path
from typing import Optional
import unicodedata

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("datasets").setLevel(logging.ERROR)

# ── Set cache dir to /kaggle/working to avoid /root quota issues ──
CACHE_DIR = "/kaggle/working/hf_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"]            = CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = f"{CACHE_DIR}/hub"
os.environ["HF_DATASETS_CACHE"]  = f"{CACHE_DIR}/datasets"
os.environ["TORCH_HOME"]         = f"{CACHE_DIR}/torch"
os.environ["HF_HUB_DISABLE_XET"] = "1"  # Disable xet protocol (Kaggle firewall)

# ── Kaggle Secrets ────────────────────────────────────────────────
# Add your secrets in Kaggle → Add-ons → Secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    HF_TOKEN      = secrets.get_secret("HF_TOKEN")
    GROQ_API_KEY  = secrets.get_secret("GROQ_API_KEY")
    print("✅ Secrets loaded from Kaggle Secrets.")
except Exception:
    # Fallback to environment variables (local testing)
    HF_TOKEN     = os.getenv("HF_TOKEN", "")
    GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
    print("⚠️  Could not load Kaggle secrets. Using env vars.")

# ── Now safe to import HF ──────────────────────────────────────────
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from huggingface_hub import login

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ Logged into Hugging Face Hub.")
else:
    print("⚠️  No HF_TOKEN set. Public models only.")

# ── Device info ────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB")
else:
    print("⚠️  Running on CPU — will be slow.")

print(f"PyTorch: {torch.__version__}")

✅ Secrets loaded from Kaggle Secrets.
✅ Logged into Hugging Face Hub.
✅ GPU: Tesla T4 | VRAM: 15.6 GB
PyTorch: 2.10.0+cu128


## 🔧 Cell 3 — Config & Hyperparameters

In [3]:
# ─────────────────────────────────────────────────────────────────
# All tunable parameters in one place.
# Modify these before running full evaluation.
# ─────────────────────────────────────────────────────────────────

CFG = {
    # ── Models ──────────────────────────────────────────────────
    "finetuned_model_id" : "ruch9265/distil-whisper-torgo",
    "baseline_model_id"  : "openai/whisper-tiny",     # Pre-FT baseline for comparison
    "llm_model"          : "llama-3.3-70b-versatile", # Groq LLM

    # ── Dataset ──────────────────────────────────────────────────
    "dataset_id"         : "abnerh/TORGO-database",
    "target_sr"          : 16000,                     # Whisper expects 16kHz

    # ── Evaluation subsets ────────────────────────────────────────
    # Run small subset first to validate pipeline, then full
    "subset_size"        : 200,     # Quick validation run (~ 8 min)
    "full_eval"          : True,    # Set False to only run subset
    "dysarthria_only"    : False,   # True = eval dysarthric samples only

    # ── LLM Gate ─────────────────────────────────────────────────
    # If ASR model's avg log-prob (proxy confidence) > threshold, skip LLM
    # This saves Groq API quota on already-correct transcriptions
    "llm_confidence_threshold" : 0.80,  # 0.0 = always use LLM, 1.0 = never

    # ── Pause detection ───────────────────────────────────────────
    "pause_threshold_sec" : 0.8,    # Gaps >= this get annotated as [Xs pause]

    # ── Batch sizes ───────────────────────────────────────────────
    # T4 has 15GB VRAM. Whisper-tiny is small; batch_size=16 is safe.
    # Reduce to 8 if you see OOM errors.
    "asr_batch_size"     : 16,

    # ── Output ────────────────────────────────────────────────────
    "output_dir"         : "/kaggle/working/results",
    "checkpoint_file"    : "/kaggle/working/results/checkpoint.json",

    # ── BERTScore model ───────────────────────────────────────────
    # distilbert is fast + fits comfortably on T4 alongside Whisper
    "bertscore_model"    : "distilbert-base-uncased",
}

os.makedirs(CFG["output_dir"], exist_ok=True)
print("✅ Config set.")
print(json.dumps({k: v for k, v in CFG.items() if "key" not in k.lower()}, indent=2))

✅ Config set.
{
  "finetuned_model_id": "ruch9265/distil-whisper-torgo",
  "baseline_model_id": "openai/whisper-tiny",
  "llm_model": "llama-3.3-70b-versatile",
  "dataset_id": "abnerh/TORGO-database",
  "target_sr": 16000,
  "subset_size": 200,
  "full_eval": true,
  "dysarthria_only": false,
  "llm_confidence_threshold": 0.8,
  "pause_threshold_sec": 0.8,
  "asr_batch_size": 16,
  "output_dir": "/kaggle/working/results",
  "checkpoint_file": "/kaggle/working/results/checkpoint.json",
  "bertscore_model": "distilbert-base-uncased"
}


## 🧹 Cell 4 — Text Preprocessing & Normalization

**Why this matters:** WER/CER are string-distance metrics. Without normalization,  
`"Hello,"` ≠ `"hello"` — inflating error rates artificially.

**Normalization pipeline applied:**
1. Unicode NFKC normalization (handles fancy quotes, ligatures)
2. Lowercase
3. Expand common contractions (`don't` → `do not`)
4. Strip punctuation (except apostrophes handled above)
5. Remove filler words (`um`, `uh`, `hmm`)
6. Collapse multiple spaces
7. Strip leading/trailing whitespace

In [4]:
import re, unicodedata, string

# ── Contraction expansion table ────────────────────────────────────
# Common English contractions → expanded form.
# Applied BEFORE punctuation stripping so apostrophes are still present.
CONTRACTIONS = {
    "i'm":"i am", "i've":"i have", "i'll":"i will", "i'd":"i would",
    "you're":"you are", "you've":"you have", "you'll":"you will", "you'd":"you would",
    "he's":"he is", "he'll":"he will", "he'd":"he would",
    "she's":"she is", "she'll":"she will", "she'd":"she would",
    "it's":"it is", "it'll":"it will",
    "we're":"we are", "we've":"we have", "we'll":"we will", "we'd":"we would",
    "they're":"they are", "they've":"they have", "they'll":"they will", "they'd":"they would",
    "that's":"that is", "that'll":"that will",
    "there's":"there is", "there're":"there are",
    "here's":"here is",
    "what's":"what is", "what're":"what are", "what'll":"what will",
    "where's":"where is", "who's":"who is", "how's":"how is",
    "don't":"do not", "doesn't":"does not", "didn't":"did not",
    "won't":"will not", "wouldn't":"would not",
    "can't":"cannot", "couldn't":"could not",
    "isn't":"is not", "aren't":"are not", "wasn't":"was not", "weren't":"were not",
    "haven't":"have not", "hasn't":"has not", "hadn't":"had not",
    "shouldn't":"should not", "mustn't":"must not",
    "let's":"let us",
    "'s":" is",  # Possessive handled below separately
}

# Filler words to remove from transcripts (ASR artifacts, not real words)
FILLERS = {"um", "uh", "uhh", "hmm", "hm", "er", "ah", "mhm"}

# Build contraction regex once (sorted longest-first to avoid partial matches)
_contraction_re = re.compile(
    r"\b(" + "|".join(re.escape(k) for k in sorted(CONTRACTIONS, key=len, reverse=True)) + r")\b"
)


def normalize_text(text: str, remove_fillers: bool = True) -> str:
    """
    Full normalization pipeline for ASR evaluation.

    Args:
        text: Raw string (hypothesis or reference)
        remove_fillers: If True, strip 'um', 'uh' etc.

    Returns:
        Normalized string suitable for WER/CER computation.
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    # Step 1: Unicode NFKC — handles ligatures, fancy quotes, fullwidth chars
    text = unicodedata.normalize("NFKC", text)

    # Step 2: Lowercase
    text = text.lower()

    # Step 3: Replace curly apostrophes with straight ones
    text = text.replace("\u2019", "'").replace("\u2018", "'")

    # Step 4: Expand contractions (must happen before punctuation removal)
    text = _contraction_re.sub(lambda m: CONTRACTIONS[m.group(0)], text)

    # Step 5: Remove pause markers like [1.2s pause] — these are pipeline artifacts
    text = re.sub(r"\[\d+\.?\d*s pause\]", "", text)

    # Step 6: Remove all punctuation (keep spaces)
    # Use translate for speed (faster than regex on large batches)
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Step 7: Remove filler words (whole words only)
    if remove_fillers:
        tokens = text.split()
        tokens = [t for t in tokens if t not in FILLERS]
        text = " ".join(tokens)

    # Step 8: Collapse multiple whitespace → single space
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ── Sanity check ────────────────────────────────────────────────────
test_cases = [
    ("I [2.5s pause] don't want to go.",   "i do not want to go"),
    ("Um, she's here!",                     "she is here"),
    ("I I want  water.",                    "i i want water"),  # Dedup is LLM's job
    ("HELLO   World",                       "hello world"),
    ("it\u2019s fine",                      "it is fine"),
]
all_pass = True
for raw, expected in test_cases:
    result = normalize_text(raw)
    status = "✅" if result == expected else "❌"
    if result != expected:
        all_pass = False
    print(f"{status}  '{raw}' → '{result}'  (expected: '{expected}')")

print("\n✅ All normalization tests passed!" if all_pass else "\n❌ Some tests failed — check logic above.")

✅  'I [2.5s pause] don't want to go.' → 'i do not want to go'  (expected: 'i do not want to go')
✅  'Um, she's here!' → 'she is here'  (expected: 'she is here')
✅  'I I want  water.' → 'i i want water'  (expected: 'i i want water')
✅  'HELLO   World' → 'hello world'  (expected: 'hello world')
✅  'it’s fine' → 'it is fine'  (expected: 'it is fine')

✅ All normalization tests passed!


## 📊 Cell 5 — Metrics Implementation

In [5]:
# ─────────────────────────────────────────────────────────────────
# Unified metrics computation.
# All functions accept lists of strings and return dicts.
# Normalization is applied INSIDE each function so it's consistent.
# ─────────────────────────────────────────────────────────────────
from jiwer import wer, cer, mer, wil
from bert_score import score as bert_score_fn
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ── Load semantic similarity model once ──────────────────────────
# all-MiniLM-L6-v2: 22M params, fast, good quality
print("Loading sentence transformer for semantic similarity...")
SEM_MODEL = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Sentence transformer loaded.")


def compute_wer_cer(references: list, hypotheses: list) -> dict:
    """
    Compute WER, CER, MER, WIL on normalized text.

    WER  = Word Error Rate       (substitutions + deletions + insertions) / ref_words
    CER  = Character Error Rate  (same at character level)
    MER  = Match Error Rate      (measures match quality)
    WIL  = Word Information Lost (1 - WIP)
    """
    # Normalize both reference and hypothesis before metric computation
    refs_norm = [normalize_text(r) for r in references]
    hyps_norm = [normalize_text(h) for h in hypotheses]

    # Filter out empty reference strings (unannotated samples)
    valid_pairs = [(r, h) for r, h in zip(refs_norm, hyps_norm) if r.strip()]
    if not valid_pairs:
        return {"wer": None, "cer": None, "mer": None, "wil": None, "n": 0}

    refs_v, hyps_v = zip(*valid_pairs)

    return {
        "wer": round(wer(list(refs_v), list(hyps_v)), 4),
        "cer": round(cer(list(refs_v), list(hyps_v)), 4),
        "mer": round(mer(list(refs_v), list(hyps_v)), 4),
        "wil": round(wil(list(refs_v), list(hyps_v)), 4),
        "n"  : len(valid_pairs),
    }


def compute_bertscore(references: list, hypotheses: list,
                      model_type: str = CFG["bertscore_model"],
                      device: str = DEVICE) -> dict:
    """
    BERTScore: semantic similarity using contextual embeddings.
    Returns Precision, Recall, F1 (mean over all samples).

    NOTE: BERTScore operates on the raw (not normalized) text
    since it benefits from punctuation/casing context.
    But we still remove pause markers to avoid confusing BERT.
    """
    # Only remove pause markers; keep casing + punctuation for BERT
    refs_clean = [re.sub(r"\[\d+\.?\d*s pause\]", "", r).strip() for r in references]
    hyps_clean = [re.sub(r"\[\d+\.?\d*s pause\]", "", h).strip() for h in hypotheses]

    # Filter empty pairs
    valid = [(r, h) for r, h in zip(refs_clean, hyps_clean) if r.strip() and h.strip()]
    if not valid:
        return {"bertscore_P": None, "bertscore_R": None, "bertscore_F1": None}

    refs_v, hyps_v = zip(*valid)

    P, R, F1 = bert_score_fn(
        list(hyps_v), list(refs_v),
        model_type=model_type,
        device=device,
        verbose=False,
        batch_size=64,
    )
    return {
        "bertscore_P"  : round(P.mean().item(), 4),
        "bertscore_R"  : round(R.mean().item(), 4),
        "bertscore_F1" : round(F1.mean().item(), 4),
    }


def compute_semantic_similarity(references: list, hypotheses: list) -> dict:
    """
    Compute mean cosine similarity between reference & hypothesis
    sentence embeddings using all-MiniLM-L6-v2.

    Unlike BERTScore (token-level), this compares full-sentence
    meaning — useful to verify the LLM repair preserves intent.
    """
    # Normalize for fair comparison
    refs_norm = [normalize_text(r) for r in references]
    hyps_norm = [normalize_text(h) for h in hypotheses]

    valid = [(r, h) for r, h in zip(refs_norm, hyps_norm) if r.strip() and h.strip()]
    if not valid:
        return {"semantic_similarity": None}

    refs_v, hyps_v = zip(*valid)

    ref_embs  = SEM_MODEL.encode(list(refs_v), batch_size=64, show_progress_bar=False)
    hyp_embs  = SEM_MODEL.encode(list(hyps_v), batch_size=64, show_progress_bar=False)

    # Compute per-pair cosine similarity, then average
    sims = [
        float(cosine_similarity(r.reshape(1, -1), h.reshape(1, -1))[0][0])
        for r, h in zip(ref_embs, hyp_embs)
    ]
    return {"semantic_similarity": round(np.mean(sims), 4)}


def compute_all_metrics(references: list, hypotheses: list,
                        phase_label: str = "") -> dict:
    """Run all metrics and return combined dict."""
    print(f"  Computing WER/CER for {phase_label}...")
    wer_cer  = compute_wer_cer(references, hypotheses)

    print(f"  Computing BERTScore for {phase_label}...")
    bscore   = compute_bertscore(references, hypotheses)

    print(f"  Computing Semantic Similarity for {phase_label}...")
    sem_sim  = compute_semantic_similarity(references, hypotheses)

    return {"phase": phase_label, **wer_cer, **bscore, **sem_sim}


print("✅ Metrics functions defined.")

Loading sentence transformer for semantic similarity...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Sentence transformer loaded.
✅ Metrics functions defined.


## 📁 Cell 6 — Load TORGO Dataset

In [6]:
# ─────────────────────────────────────────────────────────────────
# Load the TORGO dataset from HuggingFace Hub.
# Only 'train' split exists — we create our own test split.
# ─────────────────────────────────────────────────────────────────
from datasets import load_dataset, Audio

print("Loading TORGO dataset...")
print("Note: Only 'train' split exists. We'll create a held-out test split.")

full_dataset = load_dataset(
    CFG["dataset_id"],
    split="train",
    trust_remote_code=True,
    cache_dir=CFG["output_dir"],
)

print(f"✅ Loaded {len(full_dataset)} samples.")
print(f"Features: {list(full_dataset.features.keys())}")
print(f"\nClass distribution:")

# Show class distribution
import collections
status_counts = collections.Counter(full_dataset["speech_status"])
for label, count in status_counts.items():
    print(f"  {label}: {count} ({100*count/len(full_dataset):.1f}%)")

# ── Cast audio to 16kHz (the dataset is already 16kHz but cast enforces it) ──
full_dataset = full_dataset.cast_column(
    "audio",
    Audio(sampling_rate=CFG["target_sr"])
)
print(f"\n✅ Audio cast to {CFG['target_sr']} Hz.")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'abnerh/TORGO-database' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading TORGO dataset...
Note: Only 'train' split exists. We'll create a held-out test split.
✅ Loaded 16552 samples.
Features: ['audio', 'transcription', 'speech_status', 'gender', 'duration']

Class distribution:
  healthy: 10978 (66.3%)
  dysarthria: 5574 (33.7%)

✅ Audio cast to 16000 Hz.


## ✂️ Cell 7 — Create Stratified Eval Split

In [7]:
import numpy as np
import collections

EVAL_SEED = 42

# ── Step 1: Filter bad samples first (before splitting) ──────────
# Remove empty transcriptions
clean_dataset = full_dataset.filter(
    lambda x: isinstance(x["transcription"], str) and x["transcription"].strip() != ""
)
print(f"After removing empty transcriptions: {len(clean_dataset)}")

# Remove very short audio (< 0.3s) — likely silence or corrupt
clean_dataset = clean_dataset.filter(lambda x: x["duration"] >= 0.3)
print(f"After removing short audio (< 0.3s): {len(clean_dataset)}")

# ── Step 2: Manual stratified split ──────────────────────────────
# speech_status is a plain string Value column, NOT ClassLabel,
# so stratify_by_column raises ValueError. We do it manually.
def stratified_split(dataset, test_fraction, seed):
    """
    Returns (train_ds, test_ds) with test_fraction held out,
    stratified by speech_status label.
    """
    rng_local = np.random.default_rng(seed)
    all_labels = dataset["speech_status"]
    unique_labels = list(set(all_labels))

    test_indices = []
    for label in unique_labels:
        label_idx = [i for i, s in enumerate(all_labels) if s == label]
        n_test = max(1, int(len(label_idx) * test_fraction))
        chosen = rng_local.choice(label_idx, size=n_test, replace=False).tolist()
        test_indices.extend(chosen)

    test_set = set(test_indices)
    train_indices = [i for i in range(len(dataset)) if i not in test_set]

    return dataset.select(train_indices), dataset.select(test_indices)

_, eval_full = stratified_split(clean_dataset, test_fraction=0.10, seed=EVAL_SEED)
print(f"\nFull eval set size: {len(eval_full)}")

status_counts = collections.Counter(eval_full["speech_status"])
for label, count in status_counts.items():
    print(f"  {label}: {count} ({100*count/len(eval_full):.1f}%)")

# ── Step 3: Stratified subset for quick validation ────────────────
_, eval_subset = stratified_split(
    eval_full,
    test_fraction=min(CFG["subset_size"], len(eval_full)) / len(eval_full),
    seed=EVAL_SEED,
)
print(f"\nSubset for quick validation: {len(eval_subset)} samples")

subset_counts = collections.Counter(eval_subset["speech_status"])
for label, count in subset_counts.items():
    print(f"  {label}: {count}")

print("\n✅ Evaluation splits ready.")


After removing empty transcriptions: 16552
After removing short audio (< 0.3s): 16548

Full eval set size: 1654
  healthy: 1097 (66.3%)
  dysarthria: 557 (33.7%)

Subset for quick validation: 199 samples
  healthy: 132
  dysarthria: 67

✅ Evaluation splits ready.


## 🎯 Cell 8 — Load ASR Models

In [8]:
# ─────────────────────────────────────────────────────────────────
# Load both models:
#   1. openai/whisper-tiny      → baseline (no fine-tuning)
#   2. ruch9265/distil-whisper-torgo → fine-tuned on TORGO
#
# Memory strategy:
#   - Load baseline first, run inference, delete, then load FT model
#   - This avoids holding 2 models in VRAM simultaneously
#   - Each model is ~40MB on disk, ~200MB VRAM in fp32
# ─────────────────────────────────────────────────────────────────
from transformers import pipeline as hf_pipeline, WhisperFeatureExtractor


def load_asr_model(model_id: str, device: str = DEVICE) -> object:
    """
    Load a Whisper-compatible ASR model as a HF pipeline.

    Args:
        model_id: HF model repo ID
        device: 'cuda' or 'cpu'
    Returns:
        HF pipeline object
    """
    print(f"Loading {model_id}...")

    # The fine-tuned model uses 80 mel bins (whisper-tiny default)
    # Explicitly loading the feature extractor avoids config mismatch warnings
    try:
        feature_extractor = WhisperFeatureExtractor.from_pretrained(
            model_id,
            feature_size=80,
            sampling_rate=CFG["target_sr"],
        )
    except Exception:
        # Baseline whisper-tiny doesn't have custom FE config
        feature_extractor = WhisperFeatureExtractor.from_pretrained(
            model_id,
            sampling_rate=CFG["target_sr"],
        )

    # Map 'cuda' string to device index 0 for the pipeline API
    pipe_device = 0 if device == "cuda" else -1

    asr_pipe = hf_pipeline(
        "automatic-speech-recognition",
        model=model_id,
        feature_extractor=feature_extractor,
        device=pipe_device,
        torch_dtype=torch.float32,     # fp32: safe for tiny/small Whisper variants
        return_timestamps=True,        # segment-level timestamps for pause detection
        generate_kwargs={"language": "english", "task": "transcribe"},
    )

    if DEVICE == "cuda":
        vram_used = torch.cuda.memory_allocated() / 1e9
        print(f"  VRAM after load: {vram_used:.2f} GB")

    print(f"✅ {model_id} loaded.")
    return asr_pipe


def unload_model(pipe):
    """Explicitly free VRAM after a model is no longer needed."""
    del pipe
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        print(f"  VRAM after unload: {torch.cuda.memory_allocated()/1e9:.2f} GB")


print("✅ Model loading functions defined.")
print("Models will be loaded sequentially to conserve VRAM.")

✅ Model loading functions defined.
Models will be loaded sequentially to conserve VRAM.


## 🚀 Cell 9 — ASR Inference with Checkpointing

In [9]:
# ─────────────────────────────────────────────────────────────────
# Batch inference with checkpointing, confidence scoring, memory
# monitoring, and a compatibility shim for the `num_frames` KeyError
# that appears in transformers ≥ 4.45 when batching raw numpy arrays.
#
# Root cause: newer transformers changed how the ASR pipeline's
# chunked preprocessor pops `num_frames` from the feature extractor
# output. Passing audio as {"array": ..., "sampling_rate": 16000}
# dicts (instead of bare numpy arrays) routes through a different
# code path that doesn't hit the bug.
#
# We also run one sample at a time inside the loop (batch_size=1
# per pipe() call) and collect results manually — this avoids the
# DataLoader collation path that triggers the KeyError when batching.
# ─────────────────────────────────────────────────────────────────
import psutil


def estimate_confidence(result: dict) -> float:
    """
    Proxy confidence score from Whisper output.

    Whisper pipeline doesn't expose log-probs directly.
    Heuristic: fraction of chunks with non-empty text.
    Empty chunks indicate repetition/noise loops.

    Returns:
        Float in [0, 1]. Higher = more confident.
    """
    text = result.get("text", "").strip()
    if not text:
        return 0.0
    chunks = result.get("chunks", [])
    if not chunks:
        return 0.7  # Short audio with no chunks — assume OK
    non_empty = sum(1 for c in chunks if c.get("text", "").strip())
    return non_empty / len(chunks)

def safe_pipe_call(pipe, audio_array: "np.ndarray") -> dict:
    """
    Call the ASR pipeline on a single audio array safely.
    """
    MAX_SAMPLES = CFG["target_sr"] * 29  # 29s < 30s Whisper limit, safe margin

    arr = audio_array
    if len(arr) > MAX_SAMPLES:
        arr = arr[:MAX_SAMPLES]  # Hard truncate — avoids out of bounds errors

    audio_input = {
        "array": arr,
        "sampling_rate": CFG["target_sr"],
    }

    # ── FIX: Add chunk_length_s=30 to force the safe chunked processing path ──
    # This bypasses the KeyError: 'num_frames' bug in the transformers pipeline
    result = pipe(audio_input, chunk_length_s=30)
    
    return result
    
def get_audio_array(dataset, i: int) -> "np.ndarray | None":
    """
    Safely extract a float32 16kHz numpy array from a HF dataset sample.
    Falls back to librosa if the 'array' key is missing or decoding fails.
    """
    try:
        sample = dataset[i]["audio"]
        arr = sample["array"]
        
        # ── FIX: Use try-except instead of .get() for AudioDecoder objects ──
        try:
            sr = sample["sampling_rate"]
        except (KeyError, TypeError):
            sr = CFG["target_sr"]
            
        # Resample if needed (some TORGO samples are 22050 Hz)
        if sr != CFG["target_sr"]:
            import librosa
            arr = librosa.resample(arr, orig_sr=sr, target_sr=CFG["target_sr"])
        return arr.astype(np.float32)
    except (KeyError, TypeError):
        # 'array' or 'num_frames' decode error — try loading from path
        try:
            import librosa
            path = dataset[i]["audio"].get("path") or dataset[i].get("audio_filepath")
            if path:
                arr, _ = librosa.load(path, sr=CFG["target_sr"], mono=True)
                return arr.astype(np.float32)
        except Exception:
            pass
        return None
def run_asr_inference(
    dataset,
    model_id: str,
    checkpoint_key: str,
    batch_size: int = CFG["asr_batch_size"],
) -> list:
    """
    Run ASR on entire dataset with checkpointing.
    """
    checkpoint = {}
    results = []
    start_idx = 0

    # ── Load checkpoint if it exists (resume after disconnect) ────
    if os.path.exists(CFG["checkpoint_file"]):
        with open(CFG["checkpoint_file"]) as f:
            checkpoint = json.load(f)
        if checkpoint_key in checkpoint:
            results = checkpoint[checkpoint_key]
            start_idx = len(results)
            print(f"⏩ Resuming '{checkpoint_key}' from checkpoint ({start_idx} done).")
            
            # If completely finished, just return
            if start_idx >= len(dataset):
                return results

    pipe = load_asr_model(model_id)
    n = len(dataset)

    print(f"Running inference on {n - start_idx} remaining samples (1-sample-per-call mode)...")
    start_time = time.time()

    # ── FIX: Start loop from start_idx instead of 0 ───────────────
    for i in tqdm(range(start_idx, n), desc=model_id.split("/")[-1]):
        audio_array = get_audio_array(dataset, i)
        if audio_array is None:
            print(f"  ⚠️  Sample {i} skipped: audio decode failed")
            res = {"text": "", "chunks": []}
        else:
            try:
                res = safe_pipe_call(pipe, audio_array)
            except Exception as e:
                # Don't let one bad sample crash the whole run
                print(f"  ⚠️  Sample {i} failed: {e}")
                res = {"text": "", "chunks": []}

        res["confidence"] = estimate_confidence(res)
        results.append(res)

        # ── Save checkpoint every `batch_size` samples ─────────────
        if (i + 1) % batch_size == 0:
            checkpoint[checkpoint_key] = results
            with open(CFG["checkpoint_file"], "w") as f:
                json.dump(checkpoint, f)
            elapsed = time.time() - start_time
            ram_gb  = psutil.virtual_memory().used / 1e9
            vram_gb = torch.cuda.memory_allocated() / 1e9 if DEVICE == "cuda" else 0
            print(f"  [{i+1}/{n}] {elapsed:.0f}s | RAM: {ram_gb:.1f}GB | VRAM: {vram_gb:.1f}GB")

    elapsed = time.time() - start_time
    print(f"✅ Done in {elapsed:.1f}s ({elapsed/(n-start_idx)*1000:.1f}ms/sample avg).")

    # ── Final checkpoint ──────────────────────────────────────────
    checkpoint[checkpoint_key] = results
    with open(CFG["checkpoint_file"], "w") as f:
        json.dump(checkpoint, f)

    unload_model(pipe)
    return results

print("✅ Inference functions defined (num_frames-safe mode).")


✅ Inference functions defined (num_frames-safe mode).


## 🛑 Cell 10 — Pause Formatter (from ai_pipeline_finetuned.py)

In [10]:
# ─────────────────────────────────────────────────────────────────
# Identical logic to the production pipeline.
# Converts Whisper chunk output → pause-annotated string.
# ─────────────────────────────────────────────────────────────────

def format_pause_aware_transcript(
    whisper_result: dict,
    pause_threshold: float = CFG["pause_threshold_sec"],
) -> str:
    """
    Annotate long silences in Whisper output.

    If timestamp chunks are available, gaps >= pause_threshold get
    annotated as '[Xs pause]'. Otherwise returns plain text.

    Returns:
        Pause-annotated string (for LLM input)
    """
    chunks = whisper_result.get("chunks", [])

    if not chunks:
        return whisper_result.get("text", "").strip()

    def _valid_ts(chunk):
        ts = chunk.get("timestamp")
        return (
            ts is not None
            and len(ts) == 2
            and ts[0] is not None
            and ts[1] is not None
        )

    has_timestamps = any(_valid_ts(c) for c in chunks)

    if not has_timestamps:
        return " ".join(c.get("text", "").strip() for c in chunks).strip()

    parts = []
    for i, chunk in enumerate(chunks):
        word = chunk.get("text", "").strip()
        if not word:
            continue
        if i > 0 and _valid_ts(chunks[i - 1]) and _valid_ts(chunk):
            gap = chunk["timestamp"][0] - chunks[i - 1]["timestamp"][1]
            if gap >= pause_threshold:
                parts.append(f"[{gap:.1f}s pause]")
        parts.append(word)

    return " ".join(parts)


# ── Test pause formatter ─────────────────────────────────────────
mock_result = {
    "text": " I want water",
    "chunks": [
        {"text": " I",     "timestamp": (0.0, 0.5)},
        {"text": " want",  "timestamp": (1.8, 2.1)},  # 1.3s gap → annotated
        {"text": " water", "timestamp": (2.5, 3.0)},  # 0.4s gap → not annotated
    ]
}
formatted = format_pause_aware_transcript(mock_result)
print(f"Formatted: '{formatted}'")
assert "[1.3s pause]" in formatted, "Pause annotation failed!"
print("✅ Pause formatter working correctly.")

Formatted: 'I [1.3s pause] want water'
✅ Pause formatter working correctly.


## 🤖 Cell 11 — LLM Repair with Groq + Confidence Gate

In [11]:
# ─────────────────────────────────────────────────────────────────
# LLM repair using Groq API (llama-3.3-70b-versatile).
#
# Confidence gate: if ASR confidence > threshold, skip LLM
# to save API quota. The skip logic is: if the model seems
# confident AND the transcript looks complete, LLM adds noise.
# ─────────────────────────────────────────────────────────────────
from groq import Groq

# Initialize Groq client
groq_client = Groq(api_key=GROQ_API_KEY)

# ── System prompt — identical to production pipeline ──────────────
SYSTEM_PROMPT = """You are an AI Speech Transcript Repair Assistant.

You receive automatic speech recognition transcripts from a speaker with a speech impairment.

Pauses marked like [2.0s pause] may indicate where short functional words
(e.g., articles, auxiliary verbs, prepositions) were unintentionally omitted.

Your task is to minimally repair the sentence so it becomes grammatically correct
while preserving the speaker's original wording, meaning, and intent.

Repair Rules:
1. Only insert or adjust small functional words when clearly necessary
2. Do NOT paraphrase, summarize, or restructure the sentence
3. Do NOT add new information or interpretations
4. If the sentence is already correct, return it unchanged
5. Ignore pause markers in the final output
6. Remove simple speech disfluencies:
   6.1 repeated whole words (e.g., "I I want" → "I want")
   6.2 filler sounds (e.g., "um", "uh")
7. Do NOT alter emphasis or stylistic repetition used intentionally
8. Preserve original sentence boundaries; do not merge or split sentences

Output Rules:
• Output exactly one corrected sentence
• No explanations
• No commentary
• No quotation marks"""

# ── Few-shot examples (same as production) ────────────────────────
FEW_SHOT_EXAMPLES = [
    {"role": "user",      "content": "Transcript: I [2.5s pause] go [1.2s pause] store [2.0s pause] milk"},
    {"role": "assistant", "content": "I am going to the store to get some milk."},
    {"role": "user",      "content": "Transcript: Want [3.0s pause] water [1.0s pause] cold"},
    {"role": "assistant", "content": "I want a glass of cold water."},
    {"role": "user",      "content": "Transcript: Turn [1.5s pause] light off [2.5s pause] room"},
    {"role": "assistant", "content": "Please turn off the lights in the room."},
    {"role": "user",      "content": "Transcript: My name [0.5s pause] is [4.0s pause] John"},
    {"role": "assistant", "content": "My name is John."},
]


def llm_repair(transcript: str, max_retries: int = 3) -> str:
    """
    Call Groq API to repair a dysarthric transcript.

    Args:
        transcript: Pause-annotated transcript from Whisper
        max_retries: Retry on rate-limit or network error

    Returns:
        Repaired transcript string
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        *FEW_SHOT_EXAMPLES,
        {"role": "user", "content": f"Transcript: {transcript}"},
    ]

    for attempt in range(max_retries):
        try:
            response = groq_client.chat.completions.create(
                model=CFG["llm_model"],
                messages=messages,
                temperature=0.0,
                max_tokens=200,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if "rate_limit" in str(e).lower() and attempt < max_retries - 1:
                wait = 2 ** attempt  # Exponential backoff: 1s, 2s, 4s
                time.sleep(wait)
            else:
                print(f"  LLM error: {e}")
                return transcript  # Fallback: return unrepaired transcript


import re, time

def _parse_retry_after(error_message: str) -> float:
    """Extract seconds to wait from Groq 429 message."""
    # Matches '2m44.16s', '0m30s', '44.16s', '120s' etc.
    m = re.search(r'(?:(\d+)m)?(\d+(?:\.\d+)?)s', str(error_message))
    if m:
        minutes = float(m.group(1) or 0)
        seconds = float(m.group(2))
        return minutes * 60 + seconds
    return 60.0  # safe default


def process_with_llm_gate(
    asr_result: dict,
    groq_client,
    confidence_threshold: float = CFG["llm_confidence_threshold"],
    max_retries: int = 3,
) -> dict:
    confidence = asr_result.get("confidence", 0.0)
    raw_text   = asr_result.get("text", "").strip()

    if confidence >= confidence_threshold:
        return {"text": raw_text, "llm_used": False, "confidence": confidence}

    pause_text = format_pause_aware_transcript(asr_result)

    for attempt in range(max_retries):
        try:
            response = groq_client.chat.completions.create(
                model=CFG["llm_model"],
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": pause_text},
                ],
                max_tokens=256,
                temperature=0.1,
            )
            repaired = response.choices[0].message.content.strip()
            return {"text": repaired, "llm_used": True, "confidence": confidence}

        except Exception as e:
            err_str = str(e)
            is_rate_limit = "429" in err_str or "rate_limit_exceeded" in err_str
            is_daily_cap  = "tokens per day" in err_str.lower()

            if is_daily_cap:
                # TPD exhausted — no point retrying until tomorrow
                print(f"  ⛔ Daily token cap hit. Skipping LLM for remaining samples.")
                # Signal to the outer loop to stop calling LLM entirely
                raise RuntimeError("GROQ_DAILY_CAP_EXCEEDED")

            if is_rate_limit:
                wait = _parse_retry_after(err_str) + 2  # +2s buffer
                print(f"  ⏳ Rate limit. Waiting {wait:.0f}s (attempt {attempt+1}/{max_retries})...")
                time.sleep(wait)
                continue

            # Non-rate-limit error — log and fall back immediately
            print(f"  ⚠️  LLM error (non-retryable): {e}")
            break

    # All retries exhausted or non-retryable — return raw ASR output
    return {"text": raw_text, "llm_used": False, "confidence": confidence}

print("✅ LLM functions defined.")

✅ LLM functions defined.


## 🧪 Cell 12 — SUBSET Validation Run (Phase 1 + 2 + 3)

In [12]:
def run_llm_repair(
    asr_results: list,
    checkpoint_key: str,
) -> tuple[list[str], int]:
    """
    Batch wrapper around process_with_llm_gate with checkpointing
    and daily-cap early exit.
    """
    checkpoint = {}
    texts = []
    skip_count = 0
    start_idx = 0

    # ── Resume from checkpoint ────────────────────────────────────
    if os.path.exists(CFG["checkpoint_file"]):
        with open(CFG["checkpoint_file"]) as f:
            checkpoint = json.load(f)
        if checkpoint_key in checkpoint:
            cached = checkpoint[checkpoint_key]
            texts = cached["texts"]
            skip_count = cached.get("skip_count", cached.get("skipped", 0))
            start_idx = len(texts)
            print(f"⏩ Resuming '{checkpoint_key}' from checkpoint ({start_idx} done).")
            
            if start_idx >= len(asr_results):
                return texts, skip_count

    groq_client = Groq(api_key=GROQ_API_KEY)
    llm_skip_remaining = False

    # ── FIX: Slice the input array to only process remaining items ──
    remaining_results = asr_results[start_idx:]
    
    for i, asr_res in enumerate(tqdm(remaining_results, desc="LLM repair")):
        actual_idx = start_idx + i  # Keep track of true index for logging
        
        if llm_skip_remaining:
            texts.append(asr_res.get("text", "").strip())
            skip_count += 1
            continue

        try:
            out = process_with_llm_gate(asr_res, groq_client)
            texts.append(out["text"])
            if not out["llm_used"]:
                skip_count += 1

        except RuntimeError as e:
            if "GROQ_DAILY_CAP_EXCEEDED" in str(e):
                llm_skip_remaining = True
                print(f"  ⛔ Daily cap hit at sample {actual_idx}. Raw ASR used for remainder.")
                texts.append(asr_res.get("text", "").strip())
                skip_count += 1
            else:
                raise

        # ── Checkpoint every asr_batch_size samples ────────────────
        if (actual_idx + 1) % CFG["asr_batch_size"] == 0:
            checkpoint[checkpoint_key] = {"texts": texts, "skip_count": skip_count}
            with open(CFG["checkpoint_file"], "w") as f:
                json.dump(checkpoint, f)

    # Final checkpoint
    checkpoint[checkpoint_key] = {"texts": texts, "skip_count": skip_count}
    with open(CFG["checkpoint_file"], "w") as f:
        json.dump(checkpoint, f)

    print(f"✅ LLM repair done. Skipped {skip_count}/{len(asr_results)} samples.")
    return texts, skip_count

# ─────────────────────────────────────────────────────────────────
# Quick validation on eval_subset (~200 samples).
# Purpose: Verify pipeline end-to-end before full run.
# Estimated time: ~8 minutes on T4
# ─────────────────────────────────────────────────────────────────
print("=" * 60)
print("SUBSET VALIDATION RUN")
print(f"N = {len(eval_subset)} samples")
print("=" * 60)

references_subset = eval_subset["transcription"]

# ── Phase 1: Baseline (openai/whisper-tiny, no fine-tuning) ───────
print("\n[PHASE 1] Baseline whisper-tiny...")
baseline_results_sub = run_asr_inference(
    eval_subset,
    CFG["baseline_model_id"],
    checkpoint_key="baseline_subset",
)
baseline_texts_sub = [r.get("text", "").strip() for r in baseline_results_sub]
metrics_p1_sub = compute_all_metrics(references_subset, baseline_texts_sub, "Phase1_Baseline_Subset")
print(f"  WER: {metrics_p1_sub['wer']:.4f} | CER: {metrics_p1_sub['cer']:.4f}")

# ── Phase 2: Fine-tuned Whisper ───────────────────────────────────
print("\n[PHASE 2] Fine-tuned distil-whisper-torgo...")
finetuned_results_sub = run_asr_inference(
    eval_subset,
    CFG["finetuned_model_id"],
    checkpoint_key="finetuned_subset",
)
finetuned_texts_sub = [r.get("text", "").strip() for r in finetuned_results_sub]
metrics_p2_sub = compute_all_metrics(references_subset, finetuned_texts_sub, "Phase2_Finetuned_Subset")
print(f"  WER: {metrics_p2_sub['wer']:.4f} | CER: {metrics_p2_sub['cer']:.4f}")
print(f"  WER improvement: {(metrics_p1_sub['wer'] - metrics_p2_sub['wer'])*100:.2f}pp")

# ── Phase 3: Fine-tuned + LLM Repair ─────────────────────────────
print("\n[PHASE 3] Fine-tuned + LLM repair (Groq)...")
llm_texts_sub, skip_count_sub = run_llm_repair(   # ← was process_with_llm_gate
    finetuned_results_sub,
    checkpoint_key="llm_subset",
)
metrics_p3_sub = compute_all_metrics(references_subset, llm_texts_sub, "Phase3_LLM_Subset")
print(f"  WER: {metrics_p3_sub['wer']:.4f} | CER: {metrics_p3_sub['cer']:.4f}")
print(f"  BERTScore F1: {metrics_p3_sub['bertscore_F1']:.4f}")
print(f"  Semantic Sim: {metrics_p3_sub['semantic_similarity']:.4f}")

# ── Summary table ─────────────────────────────────────────────────
summary_sub = pd.DataFrame([metrics_p1_sub, metrics_p2_sub, metrics_p3_sub])
print("\n" + "=" * 60)
print("SUBSET RESULTS SUMMARY")
print("=" * 60)
print(summary_sub[["phase", "wer", "cer", "bertscore_F1", "semantic_similarity", "n"]].to_string(index=False))

SUBSET VALIDATION RUN
N = 199 samples

[PHASE 1] Baseline whisper-tiny...
⏩ Resuming 'baseline_subset' from checkpoint (199 done).
  Computing WER/CER for Phase1_Baseline_Subset...
  Computing BERTScore for Phase1_Baseline_Subset...
  Computing Semantic Similarity for Phase1_Baseline_Subset...
  WER: 1.0000 | CER: 1.0000

[PHASE 2] Fine-tuned distil-whisper-torgo...
⏩ Resuming 'finetuned_subset' from checkpoint (199 done).
  Computing WER/CER for Phase2_Finetuned_Subset...
  Computing BERTScore for Phase2_Finetuned_Subset...
  Computing Semantic Similarity for Phase2_Finetuned_Subset...
  WER: 1.0000 | CER: 1.0000
  WER improvement: 0.00pp

[PHASE 3] Fine-tuned + LLM repair (Groq)...
⏩ Resuming 'llm_subset' from checkpoint (199 done).
  Computing WER/CER for Phase3_LLM_Subset...
  Computing BERTScore for Phase3_LLM_Subset...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Computing Semantic Similarity for Phase3_LLM_Subset...
  WER: 1.0359 | CER: 1.0407
  BERTScore F1: 0.6786
  Semantic Sim: -0.0235

SUBSET RESULTS SUMMARY
                  phase    wer    cer  bertscore_F1  semantic_similarity   n
 Phase1_Baseline_Subset 1.0000 1.0000           NaN                  NaN 199
Phase2_Finetuned_Subset 1.0000 1.0000           NaN                  NaN 199
      Phase3_LLM_Subset 1.0359 1.0407        0.6786              -0.0235 199


## 📈 Cell 13 — FULL Dataset Evaluation

In [13]:
# ─────────────────────────────────────────────────────────────────
# Full evaluation on eval_full (~1655 samples).
# Only runs if CFG['full_eval'] = True.
# Estimated time: ~2.5 hours on T4
#
# ⚠️ Kaggle disconnect safety:
#   Checkpoints are saved every 500 ASR samples and every 100 LLM calls.
#   Re-run this cell after reconnect — it will pick up where it left off.
# ─────────────────────────────────────────────────────────────────

if not CFG["full_eval"]:
    print("⏭️  Full eval skipped (CFG['full_eval'] = False). Using subset results only.")
else:
    print("=" * 60)
    print("FULL DATASET EVALUATION")
    print(f"N = {len(eval_full)} samples")
    print("=" * 60)

    references_full = eval_full["transcription"]

    # Phase 1: Baseline
    print("\n[PHASE 1] Baseline...")
    baseline_results_full = run_asr_inference(
        eval_full, CFG["baseline_model_id"], checkpoint_key="baseline_full"
    )
    baseline_texts_full = [r.get("text", "").strip() for r in baseline_results_full]
    metrics_p1_full = compute_all_metrics(references_full, baseline_texts_full, "Phase1_Baseline_Full")

    # Phase 2: Fine-tuned
    print("\n[PHASE 2] Fine-tuned...")
    finetuned_results_full = run_asr_inference(
        eval_full, CFG["finetuned_model_id"], checkpoint_key="finetuned_full"
    )
    finetuned_texts_full = [r.get("text", "").strip() for r in finetuned_results_full]
    metrics_p2_full = compute_all_metrics(references_full, finetuned_texts_full, "Phase2_Finetuned_Full")

    # Phase 3: + LLM
    print("\n[PHASE 3] LLM repair...")
    llm_texts_full, skip_count_full = run_llm_repair(
        finetuned_results_full,
        checkpoint_key="llm_full",
    )
    metrics_p3_full = compute_all_metrics(references_full, llm_texts_full, "Phase3_LLM_Full")

    # Save full results
    full_results_df = pd.DataFrame({
        "reference"       : references_full,
        "baseline_hyp"    : baseline_texts_full,
        "finetuned_hyp"   : finetuned_texts_full,
        "llm_repaired"    : llm_texts_full,
        "speech_status"   : eval_full["speech_status"],
        "gender"          : eval_full["gender"],
        "duration"        : eval_full["duration"],
    })
    full_results_df.to_csv(f"{CFG['output_dir']}/full_results.csv", index=False)

    summary_full = pd.DataFrame([metrics_p1_full, metrics_p2_full, metrics_p3_full])
    print("\n" + "=" * 60)
    print("FULL RESULTS SUMMARY")
    print("=" * 60)
    print(summary_full[["phase", "wer", "cer", "bertscore_F1", "semantic_similarity", "n"]].to_string(index=False))

FULL DATASET EVALUATION
N = 1654 samples

[PHASE 1] Baseline...
⏩ Resuming 'baseline_full' from checkpoint (1376 done).
Loading openai/whisper-tiny...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


  VRAM after load: 0.25 GB
✅ openai/whisper-tiny loaded.
Running inference on 278 remaining samples (1-sample-per-call mode)...


...

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Using `chunk_length_

✅ Done in 107.2s (385.5ms/sample avg).
  VRAM after unload: 0.25 GB
  Computing WER/CER for Phase1_Baseline_Full...
  Computing BERTScore for Phase1_Baseline_Full...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Computing Semantic Similarity for Phase1_Baseline_Full...

[PHASE 2] Fine-tuned...
⏩ Resuming 'finetuned_full' from checkpoint (1654 done).
  Computing WER/CER for Phase2_Finetuned_Full...
  Computing BERTScore for Phase2_Finetuned_Full...
  Computing Semantic Similarity for Phase2_Finetuned_Full...

[PHASE 3] LLM repair...
⏩ Resuming 'llm_full' from checkpoint (1654 done).
  Computing WER/CER for Phase3_LLM_Full...
  Computing BERTScore for Phase3_LLM_Full...
  Computing Semantic Similarity for Phase3_LLM_Full...

FULL RESULTS SUMMARY
                phase    wer    cer  bertscore_F1  semantic_similarity    n
 Phase1_Baseline_Full 2.2423 2.1438        0.6808               0.2602 1654
Phase2_Finetuned_Full 1.0000 1.0000           NaN                  NaN 1654
      Phase3_LLM_Full 1.0000 1.0000           NaN                  NaN 1654


## 📊 Cell 14 — Stratified Analysis (Dysarthric vs Healthy)

In [14]:
# ─────────────────────────────────────────────────────────────────
# Break down metrics by:
#   - speech_status (healthy vs dysarthria)
#   - gender (male vs female)
#
# This is the key analysis — the LLM should help MORE on
# dysarthric speech (more errors, more pauses) than healthy.
# ─────────────────────────────────────────────────────────────────

def stratified_metrics(
    references: list,
    hypotheses: list,
    strat_labels: list,
    phase_label: str = "",
) -> pd.DataFrame:
    """
    Compute WER/CER per stratum.

    Args:
        references: Ground truth transcriptions
        hypotheses: Model outputs
        strat_labels: List of stratum labels (e.g., 'healthy', 'dysarthria')
        phase_label: Name for display

    Returns:
        DataFrame with one row per stratum
    """
    rows = []
    unique_strata = sorted(set(strat_labels))

    for stratum in unique_strata:
        idx = [i for i, s in enumerate(strat_labels) if s == stratum]
        refs_s = [references[i] for i in idx]
        hyps_s = [hypotheses[i] for i in idx]
        m = compute_wer_cer(refs_s, hyps_s)
        rows.append({
            "phase": phase_label,
            "stratum": stratum,
            "n": len(idx),
            "wer": m["wer"],
            "cer": m["cer"],
        })

    return pd.DataFrame(rows)


# ── Use subset results (always available) ────────────────────────
status_labels_sub = eval_subset["speech_status"]
gender_labels_sub = eval_subset["gender"]

print("=" * 60)
print("STRATIFIED ANALYSIS — By Speech Status (Subset)")
print("=" * 60)

for phase, hyps in [
    ("Baseline",      baseline_texts_sub),
    ("Fine-tuned",    finetuned_texts_sub),
    ("Fine-tuned+LLM", llm_texts_sub),
]:
    df = stratified_metrics(references_subset, hyps, status_labels_sub, phase)
    print(f"\n{phase}:")
    print(df.to_string(index=False))

print("\n" + "=" * 60)
print("STRATIFIED ANALYSIS — By Gender (Subset)")
print("=" * 60)

for phase, hyps in [
    ("Baseline",      baseline_texts_sub),
    ("Fine-tuned",    finetuned_texts_sub),
    ("Fine-tuned+LLM", llm_texts_sub),
]:
    df = stratified_metrics(references_subset, hyps, gender_labels_sub, phase)
    print(f"\n{phase}:")
    print(df.to_string(index=False))

STRATIFIED ANALYSIS — By Speech Status (Subset)

Baseline:
   phase    stratum   n  wer  cer
Baseline dysarthria  67  1.0  1.0
Baseline    healthy 132  1.0  1.0

Fine-tuned:
     phase    stratum   n  wer  cer
Fine-tuned dysarthria  67  1.0  1.0
Fine-tuned    healthy 132  1.0  1.0

Fine-tuned+LLM:
         phase    stratum   n    wer    cer
Fine-tuned+LLM dysarthria  67 1.0000 1.0000
Fine-tuned+LLM    healthy 132 1.0528 1.0586

STRATIFIED ANALYSIS — By Gender (Subset)

Baseline:
   phase stratum   n  wer  cer
Baseline  female  77  1.0  1.0
Baseline    male 122  1.0  1.0

Fine-tuned:
     phase stratum   n  wer  cer
Fine-tuned  female  77  1.0  1.0
Fine-tuned    male 122  1.0  1.0

Fine-tuned+LLM:
         phase stratum   n    wer    cer
Fine-tuned+LLM  female  77 1.0000 1.0000
Fine-tuned+LLM    male 122 1.0654 1.0768


## 📋 Cell 15 — Final Results Table & Export

In [15]:
# ─────────────────────────────────────────────────────────────────
# Compile all metrics into a clean comparison table.
# Export as CSV for use in papers / reports.
# ─────────────────────────────────────────────────────────────────

# Determine which results to use (subset always available)
use_full = CFG["full_eval"] and "metrics_p1_full" in dir()

metrics_to_show = [
    metrics_p1_sub if not use_full else metrics_p1_full,
    metrics_p2_sub if not use_full else metrics_p2_full,
    metrics_p3_sub if not use_full else metrics_p3_full,
]

results_df = pd.DataFrame(metrics_to_show)

# Add relative improvement columns
wer_baseline = results_df.loc[0, "wer"]
results_df["wer_rel_imp%"] = ((wer_baseline - results_df["wer"]) / wer_baseline * 100).round(2)

cer_baseline = results_df.loc[0, "cer"]
results_df["cer_rel_imp%"] = ((cer_baseline - results_df["cer"]) / cer_baseline * 100).round(2)

# Display
cols = ["phase", "n", "wer", "wer_rel_imp%", "cer", "cer_rel_imp%",
        "bertscore_F1", "semantic_similarity", "mer", "wil"]
print("\n" + "=" * 80)
print("FINAL RESULTS TABLE")
scope = "FULL" if use_full else "SUBSET"
print(f"Scope: {scope} | Normalization: NFKC + lowercase + punctuation removed + contraction expansion")
print("=" * 80)
print(results_df[cols].to_string(index=False))

# Save
out_path = f"{CFG['output_dir']}/metrics_summary.csv"
results_df.to_csv(out_path, index=False)
print(f"\n✅ Metrics saved to: {out_path}")

# Sample predictions table
sample_n = min(20, len(eval_subset))
sample_df = pd.DataFrame({
    "reference"      : list(references_subset)[:sample_n],
    "baseline"       : baseline_texts_sub[:sample_n],
    "finetuned"      : finetuned_texts_sub[:sample_n],
    "llm_repaired"   : llm_texts_sub[:sample_n],
    "speech_status"  : list(eval_subset["speech_status"])[:sample_n],
})
sample_path = f"{CFG['output_dir']}/sample_predictions.csv"
sample_df.to_csv(sample_path, index=False)
print(f"✅ Sample predictions saved to: {sample_path}")
print("\nFirst 5 predictions:")
print(sample_df.head().to_string(index=False))


FINAL RESULTS TABLE
Scope: FULL | Normalization: NFKC + lowercase + punctuation removed + contraction expansion
                phase    n    wer  wer_rel_imp%    cer  cer_rel_imp%  bertscore_F1  semantic_similarity    mer    wil
 Phase1_Baseline_Full 1654 2.2423           0.0 2.1438          0.00        0.6808               0.2602 0.9741 0.9978
Phase2_Finetuned_Full 1654 1.0000          55.4 1.0000         53.35           NaN                  NaN 1.0000 1.0000
      Phase3_LLM_Full 1654 1.0000          55.4 1.0000         53.35           NaN                  NaN 1.0000 1.0000

✅ Metrics saved to: /kaggle/working/results/metrics_summary.csv
✅ Sample predictions saved to: /kaggle/working/results/sample_predictions.csv

First 5 predictions:
reference baseline finetuned                                                                                                      llm_repaired speech_status
  quicker                    There is no text to repair. Please provide the automatic speech 

## 🎮 Cell 16 — Interactive Inference Demo

In [17]:
# ─────────────────────────────────────────────────────────────────
# Interactive demo: run the full 3-stage pipeline on a single sample.
# Great for qualitative analysis and debugging.
# ─────────────────────────────────────────────────────────────────

def demo_single_sample(sample_idx: int, dataset, use_full_pipeline: bool = True):
    """
    Run full pipeline on one sample and display all intermediate steps.

    Args:
        sample_idx: Index in dataset
        dataset: HF Dataset object
        use_full_pipeline: If True, also run LLM repair
    """
    sample = dataset[sample_idx]

    print("=" * 60)
    print(f"Sample #{sample_idx}")
    print(f"  Reference:     '{sample['transcription']}'")
    print(f"  Speech Status: {sample['speech_status']}")
    print(f"  Gender:        {sample['gender']}")
    print(f"  Duration:      {sample['duration']:.2f}s")
    print("=" * 60)

    audio_array = sample["audio"]["array"]

    # ── Stage 1: Load fine-tuned model ──────────────────────────
    demo_pipe = load_asr_model(CFG["finetuned_model_id"])

    # ── Stage 2: ASR ────────────────────────────────────────────
    asr_result = safe_pipe_call(demo_pipe, audio_array)
    asr_text   = asr_result.get("text", "").strip()
    confidence = estimate_confidence(asr_result)

    print(f"\n[Stage 2 — ASR]")
    print(f"  Raw ASR:       '{asr_text}'")
    print(f"  Confidence:    {confidence:.3f}")
    print(f"  Chunks:        {len(asr_result.get('chunks', []))}")

    # ── Stage 3: Pause formatter ─────────────────────────────────
    pause_transcript = format_pause_aware_transcript(asr_result)
    has_pauses = bool(re.search(r"\[\d+\.?\d*s pause\]", pause_transcript))

    print(f"\n[Stage 3 — Pause Formatter]")
    print(f"  Pause-annotated: '{pause_transcript}'")
    print(f"  Has pauses:      {has_pauses}")

    # ── Stage 4: LLM Gate ────────────────────────────────────────
    skip_llm = confidence >= CFG["llm_confidence_threshold"] and not has_pauses
    print(f"\n[Stage 4 — LLM Gate]")
    print(f"  Confidence >= threshold: {confidence:.3f} >= {CFG['llm_confidence_threshold']}")
    print(f"  Skip LLM: {skip_llm}")

    # ── Stage 5: LLM Repair ──────────────────────────────────────
    if use_full_pipeline and not skip_llm:
        repaired = llm_repair(pause_transcript)
    else:
        repaired = pause_transcript
        print(f"  (LLM skipped — using pause transcript as output)")

    print(f"\n[Stage 5 — Final Output]")
    print(f"  Final:         '{repaired}'")

    # ── Metrics on this single sample ────────────────────────────
    ref_norm  = normalize_text(sample["transcription"])
    asr_norm  = normalize_text(asr_text)
    llm_norm  = normalize_text(repaired)

    print(f"\n[Normalized comparison]")
    print(f"  Reference (norm): '{ref_norm}'")
    print(f"  ASR (norm):       '{asr_norm}'")
    print(f"  LLM (norm):       '{llm_norm}'")

    asr_wer = wer([ref_norm], [asr_norm])
    llm_wer = wer([ref_norm], [llm_norm])
    print(f"\n  WER (ASR):   {asr_wer:.4f}")
    print(f"  WER (LLM):   {llm_wer:.4f}")
    delta = (asr_wer - llm_wer) * 100
    print(f"  WER delta:   {delta:+.2f}pp ({'improvement' if delta > 0 else 'regression'})")

    unload_model(demo_pipe)
    return repaired


# ── Demo: Try a dysarthric sample first ─────────────────────────
# Find first dysarthric sample in subset
dysa_idx = next(
    (i for i, s in enumerate(eval_subset["speech_status"]) if s == "dysarthria"),
    0
)
print("Demo on dysarthric sample:")
_ = demo_single_sample(dysa_idx, eval_subset)

print("\n" + "─" * 60)
print("Demo on healthy sample:")
healthy_idx = next(
    (i for i, s in enumerate(eval_subset["speech_status"]) if s == "healthy"),
    1
)
_ = demo_single_sample(healthy_idx, eval_subset)

Demo on dysarthric sample:
Sample #132
  Reference:     'hem'
  Speech Status: dysarthria
  Gender:        male
  Duration:      3.12s
Loading ruch9265/distil-whisper-torgo...


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


  VRAM after load: 0.40 GB
✅ ruch9265/distil-whisper-torgo loaded.

[Stage 2 — ASR]
  Raw ASR:       'him'
  Confidence:    1.000
  Chunks:        1

[Stage 3 — Pause Formatter]
  Pause-annotated: 'him'
  Has pauses:      False

[Stage 4 — LLM Gate]
  Confidence >= threshold: 1.000 >= 0.8
  Skip LLM: True
  (LLM skipped — using pause transcript as output)

[Stage 5 — Final Output]
  Final:         'him'

[Normalized comparison]
  Reference (norm): 'hem'
  ASR (norm):       'him'
  LLM (norm):       'him'

  WER (ASR):   1.0000
  WER (LLM):   1.0000
  WER delta:   +0.00pp (regression)
  VRAM after unload: 0.40 GB

────────────────────────────────────────────────────────────
Demo on healthy sample:
Sample #0
  Reference:     'quicker'
  Speech Status: healthy
  Gender:        male
  Duration:      1.60s
Loading ruch9265/distil-whisper-torgo...


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


  VRAM after load: 0.40 GB
✅ ruch9265/distil-whisper-torgo loaded.

[Stage 2 — ASR]
  Raw ASR:       'ficker'
  Confidence:    1.000
  Chunks:        1

[Stage 3 — Pause Formatter]
  Pause-annotated: 'ficker'
  Has pauses:      False

[Stage 4 — LLM Gate]
  Confidence >= threshold: 1.000 >= 0.8
  Skip LLM: True
  (LLM skipped — using pause transcript as output)

[Stage 5 — Final Output]
  Final:         'ficker'

[Normalized comparison]
  Reference (norm): 'quicker'
  ASR (norm):       'ficker'
  LLM (norm):       'ficker'

  WER (ASR):   1.0000
  WER (LLM):   1.0000
  WER delta:   +0.00pp (regression)
  VRAM after unload: 0.40 GB


## 🔧 Cell 17 — Troubleshooting Guide

### Common Issues & Fixes

| Problem | Symptom | Fix |
|---|---|---|
| OOM on T4 | `CUDA out of memory` | Reduce `CFG['asr_batch_size']` to 8 or 4 |
| HF Hub timeout | `ConnectionError` on model download | Re-run cell; check `HF_HUB_DISABLE_XET=1` is set |
| Groq rate limit | `RateLimitError` | Increase sleep in `process_with_llm_gate` to 0.5s |
| Empty transcripts | WER = 1.0 for many samples | Check dataset loaded; verify audio cast to 16kHz |
| Checkpoint not resuming | Re-running from scratch | Check checkpoint file path; ensure `checkpoint_key` matches |
| BERTScore slow | Takes >10 min | Use `CFG['bertscore_model'] = 'distilbert-base-uncased'` (already set) |
| `KeyError: speech_status`| Filter fails | Dataset may have different column name; inspect `full_dataset.features` |

---

### VRAM Budget (T4, 15GB)
```
whisper-tiny / distil-whisper-torgo : ~200 MB
distilbert (BERTScore)              : ~250 MB
MiniLM (semantic sim)               : ~90 MB
Data tensors (batch=16, 30s audio)  : ~1 GB peak
──────────────────────────────────────────────
Total peak                          : ~2 GB  ✅ well within 15GB
```

Models are loaded/unloaded sequentially — max 1 ASR model in VRAM at a time.

### Runtime Estimates (T4)
```
Subset (200 samples):
  ASR inference × 2 models : ~4 min
  LLM repair (Groq)         : ~3 min
  Metrics computation       : ~1 min
  Total                     : ~8 min

Full eval (1655 samples):
  ASR inference × 2 models : ~30 min
  LLM repair (Groq)         : ~90 min  (rate limited at 10 calls/sec)
  Metrics computation       : ~10 min
  Total                     : ~130 min
```

In [21]:
import shutil
from IPython.display import FileLink, display

# 1. Zip the entire results folder into a single file named 'my_results.zip'
shutil.make_archive('my_results', 'zip', '/kaggle/working/results')

# 2. Generate a clickable link to download the zip file
display(FileLink('my_results.zip'))

/kaggle/working/my_results.zip

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Diagnostic cell — run if something looks wrong
# ─────────────────────────────────────────────────────────────────

print("=" * 50)
print("DIAGNOSTICS")
print("=" * 50)

# GPU status
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    used  = torch.cuda.memory_allocated() / 1e9
    print(f"VRAM: {used:.2f}GB used / {total:.2f}GB total")

# RAM status
ram = psutil.virtual_memory()
print(f"RAM:  {ram.used/1e9:.1f}GB used / {ram.total/1e9:.1f}GB total")

# Disk status
disk = psutil.disk_usage("/kaggle/working")
print(f"Disk: {disk.used/1e9:.1f}GB used / {disk.total/1e9:.1f}GB total")

# Checkpoint status
if os.path.exists(CFG["checkpoint_file"]):
    with open(CFG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    print(f"\nCheckpoint keys: {list(ckpt.keys())}")
    for k, v in ckpt.items():
        n = len(v) if isinstance(v, list) else len(v.get("texts", []))
        print(f"  {k}: {n} samples saved")
else:
    print("\nNo checkpoint file found.")

# Package versions
import transformers, datasets, jiwer
print(f"\ntransformers: {transformers.__version__}")
print(f"datasets:     {datasets.__version__}")
print(f"jiwer:        {jiwer.__version__}")
print(f"torch:        {torch.__version__}")

## 🗑️ Cell 18 — Cleanup

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Optional cleanup — run after all evaluation is complete.
# Frees VRAM and lists output files.
# ─────────────────────────────────────────────────────────────────

# Free GPU memory
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
    print(f"VRAM freed. Current usage: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# List output files
print("\nOutput files:")
for f in sorted(Path(CFG["output_dir"]).iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}  ({size_kb:.1f} KB)")

print("\n✅ Pipeline complete. All results saved to /kaggle/working/results/")